[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/049.%20The%20Causal%20%26%20Regression-Based%20Forecasting%20Problem/P49-Tier-1_executed.ipynb)

---


# 49. The Causal and Regression-Based Forecasting Problem

## Tier 1: The Pen-and-Paper Method — Multiple Linear Regression

### Purpose and scope

This notebook uses ordinary least squares (OLS), written as matrix operations, to estimate a linear relationship between port throughput and three predictors. It then evaluates in-sample fit, constructs a forecast for month 6, and examines basic regression diagnostics. The inputs are five monthly observations; the output is a fitted equation, diagnostic summaries, and a forecast with a prediction interval.

The example is deliberately small enough to work through by hand and in code. It is not enough data to establish that the model is reliable for operational forecasting. In particular, four regression parameters are estimated from five observations, leaving only one residual degree of freedom. Diagnostic tests and uncertainty estimates are consequently very fragile.

### Model assumptions and interpretation

The linear model describes conditional associations between the predictors and throughput. A regression coefficient is the estimated change in expected throughput associated with a one-unit increase in its predictor, **holding the other predictors fixed**. This interpretation is causal only when additional causal-identification conditions are justified; regression alone does not establish causation.

The usual OLS and prediction-interval interpretation relies on a correctly specified linear conditional mean, independent errors with constant variance, and (for exact small-sample inference) normally distributed errors. The design matrix must have full column rank, meaning that no predictor is an exact linear combination of the others and the intercept. Strong predictor correlation can make coefficient estimates unstable even when the matrix has full rank. Monthly observations may be dependent over time, so independence deserves particular scrutiny here.

### Example variables

The example contains five months of historical observations and three predictors:

- Trade Volume Index, denoted by $X_1$ (`Trade_Index`)
- GDP Growth Rate, denoted by $X_2$ (`GDP_Growth`)
- Fuel Price Index, denoted by $X_3$ (`Fuel_Price`)
- Container throughput in TEU, denoted by $y$ (`Throughput`)

The predictor values for month 6 are supplied separately for forecasting. No actual month 6 throughput is available for comparison.

### Solution sequence

1. Assemble the observations into a predictor matrix and response vector.
2. Add a column of ones so the model can estimate an intercept.
3. Estimate the coefficients with the OLS normal-equation formula.
4. Calculate fitted values, residuals, and in-sample performance metrics.
5. Forecast month 6 and calculate a prediction interval.
6. Inspect diagnostic statistics and plots, interpreting them cautiously given the very small sample.

## 1. The regression model and OLS estimation

A row of the data pairs one month’s predictor values with its observed throughput. The model uses a weighted sum of the predictors plus an intercept; the error term represents variation not explained by that linear relationship. Including a column of ones in the design matrix lets the same matrix equation estimate the intercept and all three slopes.

For $n$ observations and $k=3$ predictors, the model is

$$
y_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \beta_3 x_{i3} + \varepsilon_i,
\qquad i=1,\ldots,n.
$$

In matrix notation, this is $y=X\beta+\varepsilon$. The design matrix $X$ has a leading column of ones, so it has $p=k+1=4$ columns, and $\beta=(\beta_0,\beta_1,\beta_2,\beta_3)^T$. OLS chooses the coefficient vector that minimizes the sum of squared residuals:

$$
\widehat{\beta}
= \underset{b\in\mathbb{R}^{p}}{\operatorname{argmin}}\;
\sum_{i=1}^{n}(y_i-x_i^T b)^2
= \underset{b\in\mathbb{R}^{p}}{\operatorname{argmin}}\;
\|y-Xb\|_2^2.
$$

When $X$ has full column rank, the minimizer satisfies the normal equations $X^T X\widehat{\beta}=X^T y$, giving

$$
\widehat{\beta}=(X^T X)^{-1}X^T y.
$$

Here, $y$ is the vector of observed throughput, $X$ is the design matrix, and $\widehat{\beta}$ contains the estimated intercept and slopes. The code forms these matrix products explicitly in `calculate_ols_coefficients`. Forming and inverting the $p\times p$ matrix requires full rank; the normal-equation calculation has approximate cost $O(np^2+p^3)$ for $n$ observations and $p$ parameters. With five observations and four parameters, the residual degrees of freedom are $n-p=1$, so the fitted model has very little information available for assessing error variability.

In [ ]:
# NumPy and pandas build the design matrix and hold the observations; plotting and
# statistical libraries support visualization, metrics, and diagnostic checks.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import mean_squared_error, r2_score

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 2. Assemble the observations

The rows of the predictor matrix and response vector must refer to the same months in the same order. Keeping the month label in the table makes that correspondence visible; the month label itself is not used as a predictor in this example.

In [ ]:
# Historical observations for five months and three predictors.
data = {
    'Month': [1, 2, 3, 4, 5],
    'Trade_Index': [105, 108, 102, 111, 107],
    'GDP_Growth': [2.1, 2.3, 1.8, 2.7, 2.2],
    'Fuel_Price': [85, 88, 82, 90, 87],
    'Throughput': [2800, 2950, 2650, 3100, 2900]
}

df = pd.DataFrame(data)
print("Historical Data:")
print(df.to_string(index=False))

# X contains predictor columns only; y contains the response in matching row order.
X = df[['Trade_Index', 'GDP_Growth', 'Fuel_Price']].values
y = df['Throughput'].values

print(f"\nShape of predictor matrix X: {X.shape}")
print(f"Shape of response vector y: {y.shape}")

## 3. Form the design matrix

The intercept is a constant term: it is the model’s expected response when all predictors are zero. Although that condition may not be meaningful for these predictors, estimating an intercept allows the fitted plane to shift vertically rather than forcing it through the origin. The leading ones column makes the intercept part of the same matrix calculation as the slopes.

In [ ]:
# The first column of ones corresponds to the intercept coefficient beta_0.
X_with_intercept = np.column_stack([np.ones(X.shape[0]), X])

print("Design Matrix X (with intercept):")
print(X_with_intercept)

print("\nResponse Vector y:")
print(y)

## 4. Estimate the coefficients

The function below follows the normal-equation sequence directly: it forms $X^T X$ and $X^T y$, then solves the stated OLS expression by multiplying by $(X^T X)^{-1}$. The inverse exists only when the design matrix has full column rank. The coefficient for a predictor is a conditional association in the units of the response per one-unit change in that predictor, with the remaining predictors held fixed; it is not automatically a causal effect.

In [ ]:
def calculate_ols_coefficients(X, y):
    """Calculate OLS coefficients from a design matrix and response vector.

    X must include a column of ones if an intercept is required. The normal
    equation requires X to have full column rank.
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    if X.ndim != 2 or y.ndim != 1 or X.shape[0] != y.shape[0]:
        raise ValueError("X must be 2D and y must be 1D with matching rows.")
    if np.linalg.matrix_rank(X) < X.shape[1]:
        raise ValueError("The design matrix must have full column rank.")

    # The normal equations express the least-squares condition X.T @ residuals = 0.
    XtX = np.dot(X.T, X)
    XtX_inv = np.linalg.inv(XtX)
    Xty = np.dot(X.T, y)
    beta_hat = np.dot(XtX_inv, Xty)

    return beta_hat


beta_hat = calculate_ols_coefficients(X_with_intercept, y)

print("Estimated Regression Coefficients:")
print(f"β₀ (Intercept): {beta_hat[0]:.4f}")
print(f"β₁ (Trade_Index): {beta_hat[1]:.4f}")
print(f"β₂ (GDP_Growth): {beta_hat[2]:.4f}")
print(f"β₃ (Fuel_Price): {beta_hat[3]:.4f}")

print("\nFitted Regression Equation:")
print(
    f"Ŷ = {beta_hat[0]:.1f} + {beta_hat[1]:.1f}X₁ "
    f"+ {beta_hat[2]:.1f}X₂ + {beta_hat[3]:.1f}X₃"
)

## 5. Fitted values, residuals, and model fit

For each observed row, the fitted value is $\widehat{y}_i=x_i^T\widehat{\beta}$ and the residual is $e_i=y_i-\widehat{y}_i$. Residuals are the observed errors after fitting; they are used to summarize in-sample fit and to examine assumptions. Mean squared error (MSE) averages squared in-sample residuals over the number of observations, RMSE is its square root and is in TEU, and $R^2$ compares the residual sum of squares with the total variation around the sample mean. These are in-sample measures, not evidence of performance on unseen months.

In [ ]:
# Fitted values use the same design matrix and coefficient order as the OLS calculation.
y_pred = np.dot(X_with_intercept, beta_hat)

mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y, y_pred)

print("In-Sample Model Performance Metrics:")
print(f"Mean Squared Error (MSE): {mse:.2f} TEU²")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} TEU")
print(f"R-squared: {r2:.4f}")

residuals = y - y_pred
print(f"\nResiduals (observed minus fitted): {residuals}")
print(f"Mean of residuals: {np.mean(residuals):.6f} TEU")

## 6. Forecast month 6 and quantify prediction uncertainty

A forecast applies the estimated equation to the supplied month 6 predictor values. A prediction interval is wider than an interval for the mean response because it includes both uncertainty in the estimated mean and the variation of a new observation around that mean.

Under the linear-model assumptions, let $x_0$ be the new observation’s design row, including its intercept entry. The estimated residual standard deviation and the standard error for predicting a new observation are

$$
s=\sqrt{\frac{\mathrm{SSE}}{n-p}},
\qquad
\mathrm{SE}_{\mathrm{pred}}(x_0)
=s\sqrt{1+x_0^T(X^T X)^{-1}x_0},
$$

where $\mathrm{SSE}=\sum_i e_i^2$, $n$ is the number of observations, and $p$ is the number of estimated parameters, including the intercept. The term $x_0^T(X^T X)^{-1}x_0$ measures how far the new predictor combination lies, in the design’s geometry, from the information in the fitted observations; the additional $1$ accounts for new-observation error. With normally distributed independent errors and a correctly specified model, a two-sided $95\%$ prediction interval is

$$
\widehat{y}_0 \;\pm\; t_{0.975,\,n-p}\,\mathrm{SE}_{\mathrm{pred}}(x_0).
$$

The code uses the Student $t$ quantile with $n-p$ residual degrees of freedom. Here there is only one residual degree of freedom, so the interval is highly sensitive to the observed residuals and depends strongly on assumptions that cannot be assessed reliably from five observations.

In [ ]:
# The new design row begins with 1 because its first entry represents the intercept.
month6_predictors = np.array([1, 109, 2.4, 89], dtype=float)
month6_forecast = np.dot(month6_predictors, beta_hat)

print("Forecast for Month 6:")
print("Predictor values: Trade Index=109, GDP Growth=2.4%, Fuel Price=89")
print(f"Forecasted Throughput: {month6_forecast:.1f} TEU")

n = len(y)
p = X_with_intercept.shape[1]
residual_degrees_of_freedom = n - p
if residual_degrees_of_freedom <= 0:
    raise ValueError("A prediction interval requires positive residual degrees of freedom.")

# Estimate the error variance using the residual degrees of freedom, not the MSE denominator.
residual_std = np.sqrt(np.sum(residuals**2) / residual_degrees_of_freedom)
XtX_inv = np.linalg.inv(np.dot(X_with_intercept.T, X_with_intercept))
leverage = np.dot(month6_predictors, np.dot(XtX_inv, month6_predictors.T))
pred_se = residual_std * np.sqrt(1 + leverage)

confidence_level = 0.95
t_critical = stats.t.ppf(
    1 - (1 - confidence_level) / 2,
    df=residual_degrees_of_freedom
)
margin_error = t_critical * pred_se

print(
    f"95% Prediction Interval: "
    f"[{month6_forecast - margin_error:.1f}, "
    f"{month6_forecast + margin_error:.1f}] TEU"
)
print(f"Prediction Uncertainty: ±{margin_error:.1f} TEU")

## 7. Visualize the fit and forecast

The plots provide complementary views rather than a single pass/fail test. Actual-versus-fitted values show in-sample agreement; residuals versus fitted values can reveal patterns such as curvature or changing spread; the coefficient chart shows estimated slopes in their original units; and the time plot places the month 6 forecast alongside the historical series. Coefficient magnitudes should not be compared as measures of relative importance without accounting for the predictors’ different units and scales.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(
    'Multiple Linear Regression for Port Throughput Forecasting',
    fontsize=16,
    fontweight='bold'
)

# Points near the diagonal have fitted values close to their observed values.
axes[0, 0].scatter(y, y_pred, alpha=0.7, s=100, color='blue', edgecolors='black')
axes[0, 0].plot(
    [min(y), max(y)], [min(y), max(y)],
    'r--', linewidth=2, label='Perfect Prediction'
)
axes[0, 0].set_xlabel('Actual Throughput (TEU)')
axes[0, 0].set_ylabel('Fitted Throughput (TEU)')
axes[0, 0].set_title('Actual vs. Fitted Values')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# A pattern in residuals can indicate model structure not captured by the fit.
axes[0, 1].scatter(
    y_pred, residuals, alpha=0.7, s=100, color='green', edgecolors='black'
)
axes[0, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Fitted Values (TEU)')
axes[0, 1].set_ylabel('Residuals (TEU)')
axes[0, 1].set_title('Residuals vs. Fitted Values')
axes[0, 1].grid(True, alpha=0.3)

predictors = ['Trade_Index', 'GDP_Growth', 'Fuel_Price']
coefficients = beta_hat[1:]
colors = ['orange', 'purple', 'brown']
bars = axes[1, 0].bar(
    predictors, coefficients, color=colors, alpha=0.7, edgecolor='black'
)
axes[1, 0].set_xlabel('Predictor Variables')
axes[1, 0].set_ylabel('Estimated Coefficient (TEU per predictor unit)')
axes[1, 0].set_title('Estimated Regression Coefficients')
axes[1, 0].grid(True, alpha=0.3)

for bar, coef in zip(bars, coefficients):
    height = bar.get_height()
    offset = abs(height) * 0.01
    axes[1, 0].text(
        bar.get_x() + bar.get_width() / 2,
        height + (offset if height >= 0 else -offset),
        f'{coef:.2f}',
        ha='center',
        va='bottom' if height >= 0 else 'top',
        fontweight='bold'
    )

months = list(range(1, 7))
axes[1, 1].plot(
    months[:5], y, 'o-', linewidth=2, markersize=8,
    label='Actual', color='blue'
)
axes[1, 1].plot(
    months[:5], y_pred, 's--', linewidth=2, markersize=8,
    label='Fitted', color='green'
)
axes[1, 1].errorbar(
    6,
    month6_forecast,
    yerr=margin_error,
    fmt='^',
    markersize=10,
    capsize=5,
    label='Forecast with 95% Prediction Interval',
    color='red'
)
axes[1, 1].set_xlabel('Month')
axes[1, 1].set_ylabel('Throughput (TEU)')
axes[1, 1].set_title('Historical Throughput and Month 6 Forecast')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xticks(months)

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("REGRESSION ANALYSIS SUMMARY")
print("=" * 60)
print(
    f"Model Equation: Ŷ = {beta_hat[0]:.1f} + {beta_hat[1]:.1f}X₁ "
    f"+ {beta_hat[2]:.1f}X₂ + {beta_hat[3]:.1f}X₃"
)
print(f"In-sample R-squared: {r2:.4f} ({r2 * 100:.1f}% of sample variance)")
print(f"In-sample RMSE: {rmse:.2f} TEU")
print(f"Month 6 Forecast: {month6_forecast:.1f} ± {margin_error:.1f} TEU")
print("Interpret all estimates cautiously: only one residual degree of freedom is available.")
print("=" * 60)

## 8. Model diagnostics and assumption checks

Diagnostics look for evidence that the assumptions used to interpret OLS estimates and prediction intervals may be questionable. A diagnostic statistic is not proof that an assumption holds, especially with only five observations. The checks below are descriptive and should be read together with the residual plot and the data-collection context.

- **Linearity:** the fitted model assumes a linear conditional mean. A residual plot can reveal some departures, but five observations provide little diagnostic power.
- **Independence:** the Durbin–Watson statistic summarizes adjacent residual differences. Its rough interpretation is not a formal conclusion here, and monthly observations may have time dependence.
- **Constant variance:** residuals versus fitted values can be inspected for changing spread. A visual impression from this sample is weak evidence.
- **Normality:** the Shapiro–Wilk test assesses a normality null hypothesis; with so few residuals, its p-value has limited ability to identify departures.
- **Multicollinearity:** the variance inflation factor (VIF) measures how much the variance of a coefficient estimate is inflated by linear dependence among predictors. A VIF is not a causal diagnostic, and its estimate is itself uncertain in a tiny sample.

In [ ]:
print("MODEL DIAGNOSTICS AND ASSUMPTION CHECKING")
print("=" * 50)

print("1. Linearity: Assumed by the model; inspect the residual plot for visible patterns.")

def durbin_watson(residuals):
    """Calculate the Durbin-Watson statistic for adjacent residuals."""
    residuals = np.asarray(residuals, dtype=float)
    denominator = np.sum(residuals**2)
    if denominator == 0:
        return np.nan
    diff = np.diff(residuals)
    return np.sum(diff**2) / denominator


dw_stat = durbin_watson(residuals)
print(f"2. Independence (Durbin-Watson statistic): {dw_stat:.3f}")
print("   Values near 2 are often associated with less first-order residual autocorrelation;")
print("   this rough guide is not a formal test conclusion for this small sample.")

print("3. Constant variance: Inspect the residuals-versus-fitted plot.")
print("   Five observations are insufficient for a dependable visual assessment.")

shapiro_stat, shapiro_p = stats.shapiro(residuals)
print(f"4. Normality (Shapiro-Wilk): W={shapiro_stat:.3f}, p={shapiro_p:.3f}")
print("   The p-value does not establish normality; power is limited with so few observations.")

def calculate_vif(X):
    """Calculate VIF values for predictor columns in a design matrix.

    The first column is treated as the intercept. Each predictor is regressed
    on the other predictors, and VIF is 1 / (1 - R-squared).
    """
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or X.shape[1] < 2:
        raise ValueError("X must include an intercept and at least one predictor.")

    vif_values = []
    for i in range(1, X.shape[1]):
        y_i = X[:, i]
        X_others = np.delete(X, i, axis=1)

        beta_i = calculate_ols_coefficients(X_others, y_i)
        y_pred_i = np.dot(X_others, beta_i)
        r2_i = r2_score(y_i, y_pred_i)

        vif = 1 / (1 - r2_i) if r2_i < 1 else float('inf')
        vif_values.append(vif)

    return vif_values


vif_values = calculate_vif(X_with_intercept)
predictor_names = ['Trade_Index', 'GDP_Growth', 'Fuel_Price']

print("5. Multicollinearity (VIF):")
for name, vif in zip(predictor_names, vif_values):
    print(f"   {name}: {vif:.2f}")
    if vif < 5:
        print("      Below the common heuristic threshold of 5; interpret cautiously.")
    elif vif < 10:
        print("      Between common heuristic thresholds of 5 and 10.")
    else:
        print("      At or above the common heuristic threshold of 10.")

print("\n" + "=" * 50)
print("DIAGNOSTIC SUMMARY:")
print("These checks cannot establish that the OLS assumptions hold.")
print("The small sample and one residual degree of freedom require substantial caution.")
print("Regression associations alone do not establish causal effects.")
print("=" * 50)

## 9. Role, strengths, limitations, and use

### Role of Tier 1

This tier develops the mathematical foundation of regression forecasting. The matrix formulation makes the estimation objective explicit, the coefficients provide interpretable conditional associations, and the residuals connect model fit to diagnostic checks. The same framework can support statistical inference when the assumptions and data provide a defensible basis for it; this notebook focuses on estimation, a prediction interval, and basic diagnostics rather than coefficient hypothesis tests.

### Strengths

- The OLS objective and its matrix solution have clear mathematical interpretations.
- Coefficients are interpretable in terms of predictor and response units, conditional on the other predictors.
- The fitted model and residuals support established diagnostic tools.
- The calculation is transparent and practical for small, well-conditioned design matrices.

### Limitations

- A linear conditional mean may fail to represent nonlinear relationships.
- Predictor selection and feature construction require substantive knowledge.
- Multicollinearity and outliers can make estimates unstable or influential.
- Dependence over time can invalidate standard uncertainty calculations.
- A large in-sample fit does not by itself demonstrate predictive performance on future data.
- Explicitly forming and inverting $X^T X$ requires full rank and can be numerically sensitive when predictors are strongly correlated.

### Appropriate uses

This formulation is useful for learning the OLS mechanics, building an interpretable baseline, and exploring predictor–response associations when the model assumptions are plausible. Before using a forecast operationally, collect enough relevant observations to assess out-of-sample performance, investigate time dependence and data quality, and justify any causal interpretation with an appropriate study design.